## Stream Customers Data From Cloud Files to Delta Lake using Auto Loader
1. Read files from cloud storage using Auto Loader
1. Transform the dataframe to add the following columns
    -   file path: Cloud file path
    -   ingestion date: Current Timestamp
1. Write the transformed data stream to Delta Lake Table

### 1. Read files using Auto Loader

In [0]:
customers_df = (
                    spark.readStream
                         .format("cloudFiles")
                         .option("cloudFiles.format", "json")
                         .option("cloudFiles.schemaLocation", "/Volumes/gizmobox/landing/operational_data/customers_autoloader/_schema")
                         .option("cloudFiles.inferColumnTypes", "true")
                         .option("cloudFiles.schemaHints", "date_of_birth DATE, member_since DATE, created_timestamp TIMESTAMP")
                         .option("cloudFiles.schemaEvolutionMode", "rescue")
                         .load("/Volumes/gizmobox/landing/operational_data/customers_autoloader/")
)

### 2. Transform the dataframe to add the following columns
- file path: Cloud file path
- ingestion date: Current Timestamp

In [0]:
from pyspark.sql.functions import current_timestamp, col

customers_transformed_df = (
                                customers_df.withColumn("file_path", col("_metadata.file_path"))
                                            .withColumn("ingestion_date", current_timestamp())
)

### 3. Write the transformed data stream to Delta Table 

In [0]:
streaming_query = (
                    customers_transformed_df.writeStream
                        .format("delta")
                        .option("checkpointLocation", "/Volumes/gizmobox/landing/operational_data/customers_autoloader/_checkpoint_stream")
                        .option("mergeSchema", "true")
                        .toTable("gizmobox.bronze.customers_autoloader")
)

In [0]:
streaming_query.stop()

In [0]:
%sql
DROP TABLE gizmobox.bronze.customers_autoloader;

In [0]:
%sql
SELECT * FROM gizmobox.bronze.customers_autoloader;

created_timestamp,customer_id,customer_name,date_of_birth,email,member_since,telephone,_rescued_data,file_path,ingestion_date
2024-11-14T20:41:18Z,3892,Jason Bridges,2004-01-01,nicole29@gmail.com,2024-10-29,+1 8220011928,"{""source"":""WebApp"",""_file_path"":""/Volumes/gizmobox/landing/operational_data/customers_autoloader/customers_2024_11.json""}",/Volumes/gizmobox/landing/operational_data/customers_autoloader/customers_2024_11.json,2025-12-11T15:16:01.446Z
2024-11-16T01:47:25Z,1987,Amanda Alvarez,1995-10-11,eric40@outlook.com,2024-10-27,+1 9673066300,"{""source"":""WebApp"",""_file_path"":""/Volumes/gizmobox/landing/operational_data/customers_autoloader/customers_2024_11.json""}",/Volumes/gizmobox/landing/operational_data/customers_autoloader/customers_2024_11.json,2025-12-11T15:16:01.446Z
2024-11-19T14:03:09Z,5816,Amy Morales,1999-12-11,katelyn82@example.org,2024-11-04,+1 4140982719,"{""source"":""WebApp"",""_file_path"":""/Volumes/gizmobox/landing/operational_data/customers_autoloader/customers_2024_11.json""}",/Volumes/gizmobox/landing/operational_data/customers_autoloader/customers_2024_11.json,2025-12-11T15:16:01.446Z
2024-11-09T14:52:31Z,5204,Jennifer Nichols DDS,1997-02-17,joseph8@example.org,2024-11-04,+1 0383356328,"{""source"":""WebApp"",""_file_path"":""/Volumes/gizmobox/landing/operational_data/customers_autoloader/customers_2024_11.json""}",/Volumes/gizmobox/landing/operational_data/customers_autoloader/customers_2024_11.json,2025-12-11T15:16:01.446Z
2024-11-23T18:16:18Z,4468,Mrs. Roberta Salas PhD,1997-05-10,null,2024-10-25,+1 4679955118,"{""source"":""WebApp"",""_file_path"":""/Volumes/gizmobox/landing/operational_data/customers_autoloader/customers_2024_11.json""}",/Volumes/gizmobox/landing/operational_data/customers_autoloader/customers_2024_11.json,2025-12-11T15:16:01.446Z
2024-11-14T16:41:14Z,4761,Timothy Smith,1996-11-07,joshua40@yahoo.com,2024-11-03,+1 9450920858,"{""source"":""WebApp"",""_file_path"":""/Volumes/gizmobox/landing/operational_data/customers_autoloader/customers_2024_11.json""}",/Volumes/gizmobox/landing/operational_data/customers_autoloader/customers_2024_11.json,2025-12-11T15:16:01.446Z
2024-11-07T22:20:00Z,2703,Anthony Miller,1997-11-02,null,2024-11-05,+1 1586310752,"{""source"":""WebApp"",""_file_path"":""/Volumes/gizmobox/landing/operational_data/customers_autoloader/customers_2024_11.json""}",/Volumes/gizmobox/landing/operational_data/customers_autoloader/customers_2024_11.json,2025-12-11T15:16:01.446Z
2024-11-12T12:57:38Z,4914,Samantha Johnson,1997-03-26,theresa14@mail.com,2024-10-13,+1 4095003738,"{""source"":""WebApp"",""_file_path"":""/Volumes/gizmobox/landing/operational_data/customers_autoloader/customers_2024_11.json""}",/Volumes/gizmobox/landing/operational_data/customers_autoloader/customers_2024_11.json,2025-12-11T15:16:01.446Z
2024-11-08T02:26:42Z,7295,Lauren Reed,2001-08-11,jeanette48@example.org,2024-10-10,+1 0104088765,"{""source"":""WebApp"",""_file_path"":""/Volumes/gizmobox/landing/operational_data/customers_autoloader/customers_2024_11.json""}",/Volumes/gizmobox/landing/operational_data/customers_autoloader/customers_2024_11.json,2025-12-11T15:16:01.446Z
2024-11-09T20:24:30Z,7803,Katherine Lee,2000-03-09,jennifer85@mail.com,2024-10-27,+1 3119886866,"{""source"":""WebApp"",""_file_path"":""/Volumes/gizmobox/landing/operational_data/customers_autoloader/customers_2024_11.json""}",/Volumes/gizmobox/landing/operational_data/customers_autoloader/customers_2024_11.json,2025-12-11T15:16:01.446Z
